In [27]:
import os
import numpy as np
import pandas as pd

from math import radians, sin, cos, sqrt, atan2

In [28]:
# Navigate to your project
os.chdir('/workspaces/DSE3101-Project')

# Verify you're in the right place
print("Current directory:", os.getcwd())
print("Contents:", os.listdir('.'))

# Navigate to data/raw folder
os.chdir('data/raw')

Current directory: /workspaces/DSE3101-Project
Contents: ['docs', '.git', 'notebooks', '.DS_Store', 'models', 'frontend', 'requirements.txt', '.gitignore', '.vscode', 'backend', 'requirements_fixed.txt', 'data', 'onemap_all_themes_full.json', 'README.md', 'onemap_all_themes_raw.txt']


In [29]:
df = pd.read_csv('propertyguru_final.csv')

# Check to see if it loaded correctly
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 20883 entries, 0 to 20882
Data columns (total 33 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   listing_url                        20883 non-null  str    
 1   exact_address                      20880 non-null  str    
 2   price_detail                       20702 non-null  str    
 3   town_detail                        20800 non-null  str    
 4   bedrooms_detail                    20800 non-null  float64
 5   bathrooms_detail                   20161 non-null  float64
 6   area_detail                        20701 non-null  str    
 7   floor_detail                       9935 non-null   str    
 8   property_type_detail               20800 non-null  str    
 9   address_from_url                   20883 non-null  str    
 10  listing_id                         20883 non-null  int64  
 11  postal_code                        20265 non-null  float64
 12  o

In [30]:
# Count how many rows are duplicate (i.e., not the first occurrence)
num_duplicates = df.duplicated(subset=["listing_url"]).sum()
print("Number of duplicate rows:", num_duplicates)

df = df.drop_duplicates(subset=["listing_url"], keep="first")

Number of duplicate rows: 6112


In [31]:
df.shape

(14771, 33)

In [32]:
# Missing values
missing_values = df.isnull().sum()
print("Missing values per column:")
print(missing_values)

Missing values per column:
listing_url                              0
exact_address                            2
price_detail                           128
town_detail                             61
bedrooms_detail                         61
bathrooms_detail                       515
area_detail                            129
floor_detail                          7773
property_type_detail                    61
address_from_url                         0
listing_id                               0
postal_code                            616
onemap_full_address                    608
onemap_road_name                       608
latitude                               608
longitude                              608
nearest_mrt_name                       608
nearest_mrt_exit                       608
nearest_mrt_distance_m                 608
num_mrt_within_1000m                   608
nearest_clinic_name                  14623
nearest_clinic_distance_m              608
nearest_park_name          

In [33]:
critical_cols = [
    'price_detail', # important to get over/under valuation label
    'area_detail', # important to predict fair valuation price using xgboost model
    'num_amenities_within_1000m', # important to calculate SAI score
]

# Drop rows where any of the critical columns have missing values
clean_df = df.dropna(subset=critical_cols)

In [34]:
clean_df.info()

<class 'pandas.DataFrame'>
Index: 14042 entries, 0 to 20882
Data columns (total 33 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   listing_url                        14042 non-null  str    
 1   exact_address                      14042 non-null  str    
 2   price_detail                       14042 non-null  str    
 3   town_detail                        14042 non-null  str    
 4   bedrooms_detail                    14042 non-null  float64
 5   bathrooms_detail                   13664 non-null  float64
 6   area_detail                        14042 non-null  str    
 7   floor_detail                       6877 non-null   str    
 8   property_type_detail               14042 non-null  str    
 9   address_from_url                   14042 non-null  str    
 10  listing_id                         14042 non-null  int64  
 11  postal_code                        14034 non-null  float64
 12  onemap

In [35]:
clean_df['room_count'] = clean_df['bedrooms_detail'] + 1 # Living room

In [38]:
clean_df['room_count'].value_counts()

room_count
4.0     9155
3.0     2844
5.0     1753
2.0      191
6.0       76
7.0       20
1.0        2
32.0       1
Name: count, dtype: int64

In [39]:
clean_df["floor_area_sqm"] = clean_df["area_detail"].str.replace(r"\D+", "", regex=True).astype(float) * 0.092903

In [45]:
# Filter for listings with rooms <= 4 (relevant for downsizing)
mask_bedrooms = clean_df["room_count"] <= 4
mask_hdb = clean_df["listing_url"].str.contains("hdb", na=False, case=False)

final_df = clean_df[mask_bedrooms & mask_hdb]

In [46]:
final_df.info()

<class 'pandas.DataFrame'>
Index: 12192 entries, 0 to 20882
Data columns (total 35 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   listing_url                        12192 non-null  str    
 1   exact_address                      12192 non-null  str    
 2   price_detail                       12192 non-null  str    
 3   town_detail                        12192 non-null  str    
 4   bedrooms_detail                    12192 non-null  float64
 5   bathrooms_detail                   11823 non-null  float64
 6   area_detail                        12192 non-null  str    
 7   floor_detail                       6082 non-null   str    
 8   property_type_detail               12192 non-null  str    
 9   address_from_url                   12192 non-null  str    
 10  listing_id                         12192 non-null  int64  
 11  postal_code                        12185 non-null  float64
 12  onemap

In [47]:
# HDB Towns in Singapore:
# https://www.propertyguru.com.sg/singapore-property-listing/hdb/hdb-singapore-estates# 

station_to_town = {
    'ADMIRALTY MRT STATION': 'Woodlands',
    'ALJUNIED MRT STATION': 'Geylang',
    'ANG MO KIO MRT STATION': 'Ang Mo Kio',
    'BAKAU LRT STATION': 'Sengkang',
    'BANGKIT LRT STATION': 'Bukit Panjang',
    'BARTLEY MRT STATION': 'Serangoon',
    'BAYSHORE MRT STATION': 'Bedok',
    'BEAUTY WORLD MRT STATION': 'Bukit Timah',
    'BEDOK MRT STATION': 'Bedok',
    'BEDOK NORTH MRT STATION': 'Bedok',
    'BEDOK RESERVOIR MRT STATION': 'Bedok',
    'BENCOOLEN MRT STATION': 'Central Area',
    'BENDEMEER MRT STATION': 'Kallang/Whampoa',
    'BISHAN MRT STATION': 'Bishan',
    'BOON KENG MRT STATION': 'Kallang/Whampoa',
    'BOON LAY MRT STATION': 'Jurong West',
    'BRADDELL MRT STATION': 'Toa Payoh',
    'BRIGHT HILL MRT STATION': 'Bishan',
    'BUANGKOK MRT STATION': 'Sengkang',
    'BUGIS MRT STATION': 'Central Area',
    'BUKIT BATOK MRT STATION': 'Bukit Batok',
    'BUKIT GOMBAK MRT STATION': 'Bukit Batok',
    'BUKIT PANJANG LRT STATION': 'Bukit Panjang',
    'BUKIT PANJANG MRT STATION': 'Bukit Panjang',
    'BUONA VISTA MRT STATION': 'Queenstown',
    'CALDECOTT MRT STATION': 'Toa Payoh',
    'CANBERRA MRT STATION': 'Sembawang',
    'CC9': 'Central Area',
    'CHANGI AIRPORT MRT STATION': 'Tampines',
    'CHENG LIM LRT STATION': 'Sengkang',
    'CHINATOWN MRT STATION': 'Central Area',
    'CHINESE GARDEN MRT STATION': 'Jurong East',
    'CHOA CHU KANG MRT STATION': 'Choa Chu Kang',
    'CLEMENTI MRT STATION': 'Clementi',
    'COMMONWEALTH MRT STATION': 'Queenstown',
    'COMPASSVALE LRT STATION': 'Sengkang',
    'CORAL EDGE LRT STATION': 'Punggol',
    'COVE LRT STATION': 'Punggol',
    'DAKOTA MRT STATION': 'Geylang',
    'DAMAI LRT STATION': 'Punggol',
    'DOVER MRT STATION': 'Queenstown',
    'DT4': 'Central Area',
    'EUNOS MRT STATION': 'Geylang',
    'FAJAR LRT STATION': 'Bukit Panjang',
    'FARRER PARK MRT STATION': 'Kallang/Whampoa',
    'FARRER ROAD MRT STATION': 'Bukit Timah',
    'FARMWAY LRT STATION': 'Sengkang',
    'FERNVALE LRT STATION': 'Sengkang',
    'GEYLANG BAHRU MRT STATION': 'Kallang/Whampoa',
    'GREAT WORLD MRT STATION': 'Central Area',
    'HARBOURFRONT MRT STATION': 'Bukit Merah',
    'HAVELOCK MRT STATION': 'Bukit Merah',
    'HOLLAND VILLAGE MRT STATION': 'Queenstown',
    'HOUGANG MRT STATION': 'Hougang',
    'JALAN BESAR MRT STATION': 'Central Area',
    'JELAPANG LRT STATION': 'Bukit Panjang',
    'JURONG EAST MRT STATION': 'Jurong East',
    'KADALOOR LRT STATION': 'Punggol',
    'KAKI BUKIT MRT STATION': 'Bedok',
    'KALLANG MRT STATION': 'Kallang/Whampoa',
    'KANGKAR LRT STATION': 'Sengkang',
    'KATONG PARK MRT STATION': 'Marine Parade',
    'KEAT HONG LRT STATION': 'Choa Chu Kang',
    'KEMBANGAN MRT STATION': 'Bedok',
    'KHATIB MRT STATION': 'Yishun',
    'KOVAN MRT STATION': 'Hougang',
    'KUPANG LRT STATION': 'Sengkang',
    'LABRADOR PARK MRT STATION': 'Bukit Merah',
    'LAKESIDE MRT STATION': 'Jurong West',
    'LAVENDER MRT STATION': 'Kallang/Whampoa',
    'LAYAR LRT STATION': 'Sengkang',
    'LENTOR MRT STATION': 'Ang Mo Kio',
    'LITTLE INDIA MRT STATION': 'Central Area',
    'LORONG CHUAN MRT STATION': 'Serangoon',
    'MACPHERSON MRT STATION': 'Geylang',
    'MARINE PARADE MRT STATION': 'Marine Parade',
    'MARINE TERRACE MRT STATION': 'Marine Parade',
    'MARSILING MRT STATION': 'Woodlands',
    'MARYMOUNT MRT STATION': 'Bishan',
    'MATTAR MRT STATION': 'Geylang',
    'MAXWELL MRT STATION': 'Central Area',
    'MAYFLOWER MRT STATION': 'Ang Mo Kio',
    'MERIDIAN LRT STATION': 'Punggol',
    'MOUNTBATTEN MRT STATION': 'Geylang',
    'NIBONG LRT STATION': 'Punggol',
    'NICOLL HIGHWAY MRT STATION': 'Central Area',
    'NOVENA MRT STATION': 'Kallang/Whampoa',
    'OASIS LRT STATION': 'Punggol',
    'ONE-NORTH MRT STATION': 'Queenstown',
    'OUTRAM PARK MRT STATION': 'Central Area',
    'PASIR RIS MRT STATION': 'Pasir Ris',
    'PAYA LEBAR MRT STATION': 'Geylang',
    'PENDING LRT STATION': 'Bukit Panjang',
    'PETIR LRT STATION': 'Bukit Panjang',
    'PHOENIX LRT STATION': 'Bukit Panjang',
    'PIONEER MRT STATION': 'Jurong West',
    'POTONG PASIR MRT STATION': 'Toa Payoh',
    'PUNGGOL LRT STATION': 'Punggol',
    'PUNGGOL MRT STATION': 'Punggol',
    'PUNGGOL POINT LRT STATION': 'Punggol',
    'QUEENSTOWN MRT STATION': 'Queenstown',
    'RANGGUNG LRT STATION': 'Sengkang',
    'REDHILL MRT STATION': 'Bukit Merah',
    'RENJONG LRT STATION': 'Sengkang',
    'RIVIERA LRT STATION': 'Punggol',
    'ROCHOR MRT STATION': 'Central Area',
    'RUMBIA LRT STATION': 'Sengkang',
    'SAMUDERA LRT STATION': 'Punggol',
    'SEGAR LRT STATION': 'Bukit Panjang',
    'SEMBAWANG MRT STATION': 'Sembawang',
    'SENGKANG MRT STATION': 'Sengkang',
    'SENJA LRT STATION': 'Bukit Panjang',
    'SERANGOON MRT STATION': 'Serangoon',
    'SIMEI MRT STATION': 'Tampines',
    'SOO TECK LRT STATION': 'Punggol',
    'SOUTH VIEW LRT STATION': 'Choa Chu Kang',
    'SUMANG LRT STATION': 'Punggol',
    'TAI SENG MRT STATION': 'Geylang',
    'TAMPINES EAST MRT STATION': 'Tampines',
    'TAMPINES MRT STATION': 'Tampines',
    'TAMPINES WEST MRT STATION': 'Tampines',
    'TANAH MERAH MRT STATION': 'Bedok',
    'TANJONG PAGAR MRT STATION': 'Central Area',
    'TECK WHYE LRT STATION': 'Choa Chu Kang',
    'TELOK BLANGAH MRT STATION': 'Bukit Merah',
    'THANGGAM LRT STATION': 'Sengkang',
    'TIONG BAHRU MRT STATION': 'Bukit Merah',
    'TOA PAYOH MRT STATION': 'Toa Payoh',
    'TONGKANG LRT STATION': 'Sengkang',
    'UBI MRT STATION': 'Geylang',
    'UPPER CHANGI MRT STATION': 'Tampines',
    'UPPER THOMSON MRT STATION': 'Bishan',
    'WOODLANDS MRT STATION': 'Woodlands',
    'WOODLANDS NORTH MRT STATION': 'Woodlands',
    'WOODLANDS SOUTH MRT STATION': 'Woodlands',
    'WOODLEIGH MRT STATION': 'Toa Payoh',
    'YEW TEE MRT STATION': 'Choa Chu Kang',
    'YIO CHU KANG MRT STATION': 'Ang Mo Kio',
    'YISHUN MRT STATION': 'Yishun'
}

In [48]:
final_df = final_df.dropna(subset=['nearest_mrt_name'])
final_df['hdb_town'] = final_df['nearest_mrt_name'].map(station_to_town)

# To check if any stations were missed (print rows where the mapping failed)
missing_towns = final_df[final_df['hdb_town'].isna()]
if not missing_towns.empty:
    print("Warning: Some stations were not mapped!")
    print(missing_towns['nearest_mrt_name'].unique())
else:
    print("All stations successfully mapped to an HDB town!")

All stations successfully mapped to an HDB town!


In [49]:
final_df['hdb_town'].value_counts()

hdb_town
Sengkang           1235
Punggol             911
Tampines            814
Yishun              795
Jurong West         698
Bukit Batok         682
Woodlands           608
Ang Mo Kio          593
Bedok               585
Toa Payoh           581
Bukit Merah         518
Queenstown          500
Hougang             485
Choa Chu Kang       398
Kallang/Whampoa     374
Bukit Panjang       364
Clementi            310
Sembawang           308
Geylang             290
Bishan              241
Jurong East         234
Pasir Ris           225
Central Area        167
Serangoon           149
Marine Parade        89
Bukit Timah          38
Name: count, dtype: int64

In [50]:
# Remove duplicate rows if any
print(final_df.duplicated().sum())
final_df = final_df.drop_duplicates()

0


In [54]:
# Amenity saturation thresholds (max counts)

# These values cap the maximum number of amenities counted towards the SAI score.
# This serves two main purposes:
# 1. Diminishing Returns: Having 3 MRTs nearby is excellent, but a 4th doesn't add much practical value.
# 2. Score Normalization: It prevents a location with an extreme abundance of one amenity 
#    (e.g., 20 clinics in a commercial hub) from disproportionately skewing the final SAI score.

max_counts = {
    'clinic': 10,
    'hawker': 5,
    'park': 3,
    'mrt': 3
}

In [55]:
import math
import pandas as pd

def calculate_sai_for_row(row, sliders, half_life_distance=500):
    """
    Calculates the SAI using a lenient Exponential Decay for distance for a single DataFrame row.
    Slider weights for 'clinic', 'hawker', 'park', 'mrt' should be between 1 and 10.
    The final SAI score is guaranteed to be between 0 and 100.
    """
    decay_rate = math.log(2) / half_life_distance
    
    # Extract distances from the row (default to 500m if missing or NaN)
    distances = {
        'clinic': row.get('nearest_clinic_distance_m', 500),
        'hawker': row.get('nearest_hawker_distance_m', 500),
        'park': row.get('nearest_park_distance_m', 500),
        'mrt': row.get('nearest_mrt_distance_m', 500)
    }
    
    # Extract counts from the row (default to 1 if missing or NaN)
    counts = {
        'clinic': row.get('num_clinic_within_1000m', 1),
        'hawker': row.get('num_hawker_within_1000m', 1),
        'park': row.get('num_park_within_1000m', 1),
        'mrt': row.get('num_mrt_within_1000m', 1) 
    }
    
    weighted_sum = 0
    total_weights = 0
    
    for category in ['clinic', 'hawker', 'park', 'mrt']:
        max_c = max_counts.get(category, 1)
        
        # Handle potential NaN values safely and ensure no negative values
        dist = distances[category] if pd.notna(distances[category]) else 500
        dist = max(0, dist) # Prevent negative distances from inflating score > 50
        
        count = counts[category] if pd.notna(counts[category]) else 1
        count = max(0, count) # Prevent negative counts
        
        # Get the slider weight (expecting 1-10, default to 1 if missing)
        weight = sliders.get(category, 1)
        
        # Cap count to max_c to ensure count_score doesn't exceed 50
        count_capped = min(count, max_c)
        
        # 1. Lenient Exponential Decay for Distance (Max 50 points)
        dist_score = 50 * math.exp(-decay_rate * dist)
        
        # 2. Linear scale for Density (Max 50 points)
        count_score = 50 * (count_capped / max_c) if max_c > 0 else 0
        
        # Total Category Score (Max 100 points)
        score = dist_score + count_score
        
        # Applying the individual slider weight to the category score
        weighted_sum += (score * weight)
        # Keeping track of the total slider values used
        total_weights += weight
        
    if total_weights == 0:
        return 0.0

    # Normalization of SAI score     
    final_sai = weighted_sum / total_weights
    
    # Strictly bound the final score between 0 and 100
    final_sai = max(0.0, min(100.0, final_sai))
    
    return round(final_sai, 1)

In [ ]:
cleaned_df['price'].info()

<class 'pandas.Series'>
Index: 317 entries, 0 to 455
Series name: price
Non-Null Count  Dtype
--------------  -----
317 non-null    str  
dtypes: str(1)
memory usage: 5.0 KB


In [ ]:
# ==========================================
# DATAFRAME APPLICATION EXAMPLE
# ==========================================

# 2. Define user sliders mapped to the new categories
# user_sliders = {
#     'clinic': 7, 
#     'hawker': 8, 
#     'park': 4, 
#     'mrt': 9
# }

import re 

def final_output(df, town, min_rooms, user_sliders, budget=None):
    """
    Filters the dataframe based on town, minimum room count, and an optional budget.
    """
    # Convert to nullable integer ('Int64') to safely drop the .0 while ignoring NaNs
    # Convert to string
    df['postal_code'] = df['postal_code'].astype('Int64').astype(str)

    # When pandas converts missing 'Int64' values to strings, it writes "<NA>". 
    # Replace those with empty strings.
    df['postal_code'] = df['postal_code'].replace('<NA>', '')

    clean_numeric_price = df['price'].replace(r'\D', '', regex=True).astype(int)
    df['formatted_price'] = clean_numeric_price.map('${:,.0f}'.format)

    # 1. Filter by town (making it case-insensitive for robustness)
    filtered_df = df[df['hdb_town'].str.lower() == town.lower()]
    
    # 2. Filter by minimum number of rooms
    filtered_df = filtered_df[filtered_df['room_count'] >= min_rooms]
    
    # Filter by budget if it is provided
    filtered_df['price'] = filtered_df['price'].replace(r'\D', '', regex=True).astype(int)
    if budget is not None:
        filtered_df = filtered_df[filtered_df['price'] <= budget]
        
    # 3. Apply the function across the dataframe (axis=1 means row-by-row)
    filtered_df['SAI_Score'] = filtered_df.apply(lambda row: calculate_sai_for_row(row, user_sliders), axis=1)

    # 4. Rank by SAI_Score in descending order
    ranked_df = filtered_df.sort_values(by='SAI_Score', ascending=False)
    
    # 5. Get the top 3 rows
    top_3_df = ranked_df.head(3)
    
    # 6. Select and return only the requested columns
    columns_to_return = ['listing_url', 'address_from_url', 'town', 'postal_code', 
                         'formatted_price', 'room_count', 'SAI_Score',
                         'nearest_mrt_name', 'nearest_mrt_distance_m', 'num_mrt_within_1000m', 
                         'nearest_clinic_distance_m', 'num_clinic_within_1000m', 
                         'nearest_community_club_name', 'nearest_community_club_distance_m', 
                         'nearest_hawker_distance_m', 'num_hawker_within_1000m', 
                         'num_park_within_1000m', 'num_park_within_1000m'
]
    
    
    return top_3_df[columns_to_return]

In [ ]:
# Test Case 1: Filter by town and min_rooms only (no budget)
user_sliders = {
    'clinic': 8, 
    'hawker': 5, 
    'park': 7, 
    'mrt': 10, 
}

print("Test Case 1: Tampines, Min 3 rooms, No budget limit")
result_1 = final_output(cleaned_df, town='Tampines', min_rooms=3, user_sliders=user_sliders)
result_1

Test Case 1: Tampines, Min 3 rooms, No budget limit


,listing_url,address_from_url,town,postal_code,formatted_price,room_count,SAI_Score,nearest_mrt_name,nearest_mrt_distance_m,num_mrt_within_1000m,nearest_clinic_distance_m,num_clinic_within_1000m,nearest_community_club_name,nearest_community_club_distance_m,nearest_hawker_distance_m,num_hawker_within_1000m,num_park_within_1000m,num_park_within_1000m
198,https://www.propertyguru.com.sg/listing/hdb-fo...,236 Tampines Street 21,Clementi,520236,"$590,000",3,64.6,TAMPINES MRT STATION,370.6,2.0,207.6,33.0,Tampines North CC (U/C),186.8,1206.0,0.0,3.0,3.0
423,https://www.propertyguru.com.sg/listing/hdb-fo...,431 Tampines Street 41,Clementi,520431,"$915,888",5,63.4,TAMPINES EAST MRT STATION,546.9,2.0,142.0,29.0,Tampines North CC (U/C),520.9,1711.8,0.0,3.0,3.0
191,https://www.propertyguru.com.sg/listing/hdb-fo...,426 Tampines Street 41,Clementi,520426,"$908,888",5,62.5,TAMPINES MRT STATION,703.2,2.0,234.6,28.0,Tampines North CC (U/C),285.5,1612.7,0.0,3.0,3.0


In [ ]:
# Test Case 2: Filter by town, min_rooms, and budget
print("Test Case 2: Tampines, Min 4 rooms, Budget <= 600,000")
result_2 = final_output(cleaned_df, town='Tampines', min_rooms=3, budget=600000, user_sliders=user_sliders)
result_2

Test Case 2: Tampines, Min 4 rooms, Budget <= 600,000


,listing_url,address_from_url,town,postal_code,formatted_price,room_count,SAI_Score,nearest_mrt_name,nearest_mrt_distance_m,num_mrt_within_1000m,nearest_clinic_distance_m,num_clinic_within_1000m,nearest_community_club_name,nearest_community_club_distance_m,nearest_hawker_distance_m,num_hawker_within_1000m,num_park_within_1000m,num_park_within_1000m
198,https://www.propertyguru.com.sg/listing/hdb-fo...,236 Tampines Street 21,Clementi,520236,"$590,000",3,64.6,TAMPINES MRT STATION,370.6,2.0,207.6,33.0,Tampines North CC (U/C),186.8,1206.0,0.0,3.0,3.0
